# 02 Compare OpenICU dyn vs YAIB/RICU reference

Use this once you have a reference `dyn.parquet` from YAIB or direct R/RICU.

In [ ]:
from pathlib import Path
import polars as pl

from openicu_yaib.concepts import DYNAMIC_VARS
from openicu_yaib.compare import (
    scan_dyn,
    normalize_reference_columns,
    schema_report,
    table_summary,
    key_overlap_report,
    stay_overlap_report,
    coverage_report,
    missingness_report,
    value_diff_report,
    worst_examples,
    reference_only_values,
    openicu_only_values,
)

In [ ]:
OPENICU_DYN = Path("~/output/openicu_dyn.parquet")
REFERENCE_DYN = Path("~/output/openicu_yaib/ricu_dynamic_vars_miiv.parquet")
REPORT_DIR = Path("~/output/openicu_vs_reference_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

openicu = scan_dyn(OPENICU_DYN)
reference = scan_dyn(REFERENCE_DYN)

# If the RICU export has charttime instead of time, use:
# reference = normalize_reference_columns(reference, id_col="stay_id", time_col="time")

## Schema

In [ ]:
schema_report(openicu, reference, DYNAMIC_VARS)

## Table summaries

In [ ]:
pl.concat([table_summary(openicu, "openicu"), table_summary(reference, "reference")])

## Stay/key overlap

In [ ]:
stay_overlap_report(openicu, reference)

In [ ]:
key_overlap_report(openicu, reference)

## Coverage

In [ ]:
coverage = coverage_report(openicu, reference, DYNAMIC_VARS)
coverage.sort("diff_non_null")

## Missingness on common stay/time keys

In [ ]:
missingness = missingness_report(openicu, reference, DYNAMIC_VARS)
missingness.sort("only_reference", descending=True)

## Numeric value differences

In [ ]:
diffs = value_diff_report(openicu, reference, DYNAMIC_VARS)
diffs.sort("max_abs_diff", descending=True)

## Worst examples

In [ ]:
worst_examples(openicu, reference, "hr", n=20)

In [ ]:
worst_examples(openicu, reference, "crea", n=20)

In [ ]:
worst_examples(openicu, reference, "urine", n=20)

## Reference-only values

In [ ]:
ref_only = reference_only_values(openicu, reference, DYNAMIC_VARS)
ref_only.group_by("concept").agg(pl.len().alias("n_reference_only")).sort("n_reference_only", descending=True).collect()

In [ ]:
target_stay = 39999858
ref_only.filter(pl.col("stay_id") == target_stay).sort(["time", "concept"]).collect()

## Save reports

In [ ]:
coverage.write_csv(REPORT_DIR / "coverage.csv")
missingness.write_csv(REPORT_DIR / "missingness.csv")
diffs.write_csv(REPORT_DIR / "value_diff.csv")
key_overlap_report(openicu, reference).write_csv(REPORT_DIR / "key_overlap.csv")
stay_overlap_report(openicu, reference).write_csv(REPORT_DIR / "stay_overlap.csv")
REPORT_DIR